# PrivAware v2 — Day 5: Chrome Extension

**Goal:** Build the complete Chrome extension that calls your live HuggingFace API and shows a risk score in the popup.

**This notebook generates all the extension files for you.**
Run every cell → download the zip at the end → load it in Chrome.

**No GPU needed today** — Runtime type can stay as CPU.

## Cell 1 — Create the extension folder structure

In [ ]:
import os
os.makedirs('/content/privaware-extension/icons', exist_ok=True)
print('Folder structure created!')
print('/content/privaware-extension/')
print('  icons/')
print('  manifest.json')
print('  popup.html')
print('  popup.js')
print('  background.js')
print('  content.js')

## Cell 2 — Write manifest.json

In [ ]:
manifest = '''{
  "manifest_version": 3,
  "name": "PrivAware",
  "version": "2.0",
  "description": "AI-powered privacy inspector. Detects phishing URLs and deceptive privacy policies using custom NLP models.",
  "permissions": [
    "activeTab",
    "storage",
    "scripting"
  ],
  "host_permissions": [
    "<all_urls>"
  ],
  "action": {
    "default_popup": "popup.html",
    "default_icon": {
      "16": "icons/icon16.png",
      "48": "icons/icon48.png",
      "128": "icons/icon128.png"
    }
  },
  "background": {
    "service_worker": "background.js"
  },
  "content_scripts": [
    {
      "matches": ["<all_urls>"],
      "js": ["content.js"]
    }
  ],
  "icons": {
    "16": "icons/icon16.png",
    "48": "icons/icon48.png",
    "128": "icons/icon128.png"
  }
}'''

with open('/content/privaware-extension/manifest.json', 'w') as f:
    f.write(manifest)
print('manifest.json written!')

## Cell 3 — Write popup.html (the UI)

In [ ]:
popup_html = '''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>PrivAware</title>
  <style>
    * { margin: 0; padding: 0; box-sizing: border-box; }
    body {
      width: 340px;
      min-height: 480px;
      background: #0f0f1a;
      color: #e2e8f0;
      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    }
    .header {
      background: #1a1a2e;
      padding: 14px 16px;
      display: flex;
      align-items: center;
      justify-content: space-between;
      border-bottom: 1px solid #2a2a4a;
    }
    .logo {
      display: flex;
      align-items: center;
      gap: 8px;
    }
    .logo-icon {
      width: 24px; height: 24px;
      background: #6c63ff;
      border-radius: 6px;
      display: flex; align-items: center; justify-content: center;
      font-size: 14px;
    }
    .logo-text { font-size: 15px; font-weight: 600; color: #fff; }
    .version { font-size: 11px; color: #666; }
    .site-bar {
      padding: 10px 16px 6px;
      background: #12122a;
      border-bottom: 1px solid #1e1e3a;
    }
    .site-label { font-size: 10px; color: #666; margin-bottom: 2px; }
    .site-url { font-size: 12px; color: #a0aec0; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
    .score-section {
      padding: 16px;
      display: flex;
      align-items: center;
      gap: 16px;
    }
    .score-ring { position: relative; width: 80px; height: 80px; flex-shrink: 0; }
    .score-ring svg { transform: rotate(-90deg); }
    .score-center {
      position: absolute; inset: 0;
      display: flex; flex-direction: column;
      align-items: center; justify-content: center;
    }
    .score-number { font-size: 22px; font-weight: 700; }
    .score-max { font-size: 9px; color: #666; }
    .score-info { flex: 1; }
    .risk-label { font-size: 16px; font-weight: 600; margin-bottom: 4px; }
    .risk-detail { font-size: 11px; color: #888; line-height: 1.6; }
    .models-section { padding: 0 16px 12px; display: flex; flex-direction: column; gap: 6px; }
    .model-card {
      background: #12122a;
      border: 1px solid #2a2a4a;
      border-radius: 8px;
      padding: 10px 12px;
    }
    .model-header { display: flex; justify-content: space-between; align-items: center; margin-bottom: 6px; }
    .model-name { font-size: 12px; font-weight: 500; color: #ccc; }
    .model-badge {
      font-size: 10px; padding: 2px 8px;
      border-radius: 20px; font-weight: 500;
    }
    .badge-safe { background: #064e3b22; color: #10b981; }
    .badge-risky { background: #78350f22; color: #f59e0b; }
    .badge-danger { background: #7f1d1d22; color: #ef4444; }
    .model-bar { height: 4px; background: #2a2a4a; border-radius: 2px; overflow: hidden; margin-bottom: 4px; }
    .model-bar-fill { height: 100%; border-radius: 2px; transition: width 0.6s ease; }
    .model-meta { font-size: 10px; color: #555; }
    .trackers-section { padding: 0 16px 12px; }
    .section-title { font-size: 11px; color: #888; margin-bottom: 6px; }
    .tracker-tags { display: flex; gap: 6px; flex-wrap: wrap; }
    .tag {
      font-size: 10px; padding: 3px 8px;
      border-radius: 20px;
    }
    .tag-ads     { background: #7f1d1d22; color: #ef4444; }
    .tag-analytics { background: #312e8122; color: #a78bfa; }
    .tag-finger  { background: #78350f22; color: #f59e0b; }
    .tag-social  { background: #064e3b22; color: #10b981; }
    .explanation {
      margin: 0 16px 14px;
      background: #0d1117;
      border-radius: 8px;
      padding: 10px 12px;
      border-left: 3px solid #6c63ff;
    }
    .explanation-label { font-size: 10px; color: #666; margin-bottom: 4px; }
    .explanation-text { font-size: 11px; color: #aaa; line-height: 1.5; }
    .footer {
      background: #12122a;
      padding: 10px 16px;
      display: flex;
      justify-content: space-between;
      align-items: center;
      border-top: 1px solid #2a2a4a;
    }
    .footer-time { font-size: 10px; color: #555; }
    .footer-link { font-size: 11px; color: #6c63ff; cursor: pointer; text-decoration: none; }
    .loading {
      display: flex; flex-direction: column;
      align-items: center; justify-content: center;
      padding: 40px 20px; text-align: center;
    }
    .spinner {
      width: 32px; height: 32px;
      border: 3px solid #2a2a4a;
      border-top-color: #6c63ff;
      border-radius: 50%;
      animation: spin 0.8s linear infinite;
      margin-bottom: 12px;
    }
    @keyframes spin { to { transform: rotate(360deg); } }
    .loading-text { font-size: 13px; color: #888; }
    .error-box {
      margin: 16px;
      background: #7f1d1d22;
      border: 1px solid #7f1d1d44;
      border-radius: 8px;
      padding: 12px;
      font-size: 12px;
      color: #ef4444;
      text-align: center;
    }
    .scan-btn {
      display: block; width: calc(100% - 32px);
      margin: 12px 16px;
      padding: 10px;
      background: #6c63ff;
      color: white;
      border: none;
      border-radius: 8px;
      font-size: 13px;
      font-weight: 600;
      cursor: pointer;
    }
    .scan-btn:hover { background: #5a52d5; }
    #main-content { display: none; }
  </style>
</head>
<body>
  <div class="header">
    <div class="logo">
      <div class="logo-icon">🛡️</div>
      <span class="logo-text">PrivAware</span>
    </div>
    <span class="version">v2.0</span>
  </div>

  <div class="site-bar">
    <div class="site-label">Analyzing</div>
    <div class="site-url" id="current-url">Loading...</div>
  </div>

  <!-- Loading state -->
  <div class="loading" id="loading-state">
    <div class="spinner"></div>
    <div class="loading-text">Scanning with AI models...</div>
  </div>

  <!-- Error state -->
  <div class="error-box" id="error-state" style="display:none">
    Could not scan this page. Try refreshing.
  </div>

  <!-- Main content -->
  <div id="main-content">
    <div class="score-section">
      <div class="score-ring">
        <svg width="80" height="80" viewBox="0 0 80 80">
          <circle cx="40" cy="40" r="33" fill="none" stroke="#2a2a4a" stroke-width="7"/>
          <circle id="score-circle" cx="40" cy="40" r="33" fill="none"
            stroke="#ef4444" stroke-width="7"
            stroke-dasharray="207"
            stroke-dashoffset="52"
            stroke-linecap="round"/>
        </svg>
        <div class="score-center">
          <span class="score-number" id="score-number">--</span>
          <span class="score-max">/100</span>
        </div>
      </div>
      <div class="score-info">
        <div class="risk-label" id="risk-label">Scanning...</div>
        <div class="risk-detail" id="risk-detail">Please wait</div>
      </div>
    </div>

    <div class="models-section">
      <div class="model-card">
        <div class="model-header">
          <span class="model-name">Phishing detector</span>
          <span class="model-badge" id="phishing-badge">—</span>
        </div>
        <div class="model-bar"><div class="model-bar-fill" id="phishing-bar" style="width:0%;background:#ef4444"></div></div>
        <div class="model-meta" id="phishing-meta">DistilBERT · v3d4nt7/privaware-phishing-detector</div>
      </div>
      <div class="model-card">
        <div class="model-header">
          <span class="model-name">Privacy policy</span>
          <span class="model-badge" id="policy-badge">—</span>
        </div>
        <div class="model-bar"><div class="model-bar-fill" id="policy-bar" style="width:0%;background:#f59e0b"></div></div>
        <div class="model-meta" id="policy-meta">DistilBERT · v3d4nt7/privaware-policy-classifier</div>
      </div>
    </div>

    <div class="trackers-section">
      <div class="section-title">Trackers detected</div>
      <div class="tracker-tags" id="tracker-tags">
        <span class="tag tag-ads">Loading...</span>
      </div>
    </div>

    <div class="explanation">
      <div class="explanation-label">AI summary</div>
      <div class="explanation-text" id="explanation-text">Analyzing...</div>
    </div>

    <button class="scan-btn" id="rescan-btn">Rescan this page</button>
  </div>

  <div class="footer">
    <span class="footer-time" id="scan-time">—</span>
    <a class="footer-link" id="history-link">View history →</a>
  </div>

  <script src="popup.js"></script>
</body>
</html>'''

with open('/content/privaware-extension/popup.html', 'w') as f:
    f.write(popup_html)
print('popup.html written!')

## Cell 4 — Write popup.js (the logic)

In [ ]:
popup_js = '''const API_URL = 'https://V3d4nt7-privaware-api.hf.space';

const TRACKERS = {
  ads: ['doubleclick', 'googlesyndication', 'adnxs', 'openx', 'rubiconproject', 'pubmatic'],
  analytics: ['google-analytics', 'hotjar', 'mixpanel', 'amplitude', 'segment', 'clarity'],
  fingerprint: ['fingerprintjs', 'fpjs', 'threatmetrix', 'iovation'],
  social: ['facebook.net', 'platform.twitter', 'connect.facebook', 'linkedin.com/analytics']
};

function detectTrackers(url) {
  const found = { ads: 0, analytics: 0, fingerprint: 0, social: 0 };
  const hostname = url.toLowerCase();
  for (const [cat, domains] of Object.entries(TRACKERS)) {
    for (const d of domains) {
      if (hostname.includes(d)) found[cat]++;
    }
  }
  return found;
}

function getRiskColor(score) {
  if (score >= 70) return '#ef4444';
  if (score >= 40) return '#f59e0b';
  return '#10b981';
}

function getRiskLabel(score) {
  if (score >= 70) return 'High Risk';
  if (score >= 40) return 'Moderate Risk';
  return 'Low Risk';
}

function getBadgeClass(label) {
  if (!label) return 'badge-risky';
  const l = label.toUpperCase();
  if (l === 'PHISHING' || l === 'DECEPTIVE') return 'badge-danger';
  if (l === 'RISKY') return 'badge-risky';
  return 'badge-safe';
}

function updateRing(score) {
  const circle = document.getElementById('score-circle');
  const circumference = 207;
  const offset = circumference - (score / 100) * circumference;
  circle.style.strokeDashoffset = offset;
  circle.style.stroke = getRiskColor(score);
}

function saveToHistory(data) {
  chrome.storage.local.get(['scanHistory'], (result) => {
    const history = result.scanHistory || [];
    history.unshift(data);
    if (history.length > 10) history.pop();
    chrome.storage.local.set({ scanHistory: history });
  });
}

async function scanPage(url) {
  document.getElementById('loading-state').style.display = 'flex';
  document.getElementById('main-content').style.display = 'none';
  document.getElementById('error-state').style.display = 'none';
  document.getElementById('current-url').textContent = url;

  try {
    const [urlResult] = await Promise.all([
      fetch(`${API_URL}/scan-url`, {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ url })
      }).then(r => r.json())
    ]);

    const trackers = detectTrackers(url);
    const trackerCount = Object.values(trackers).reduce((a, b) => a + b, 0);
    const trackerScore = Math.min(trackerCount * 10, 40);
    const combinedScore = Math.round((urlResult.risk_score * 0.6) + (trackerScore * 0.4));

    // Update UI
    document.getElementById('loading-state').style.display = 'none';
    document.getElementById('main-content').style.display = 'block';

    document.getElementById('score-number').textContent = combinedScore;
    document.getElementById('score-number').style.color = getRiskColor(combinedScore);
    document.getElementById('risk-label').textContent = getRiskLabel(combinedScore);
    document.getElementById('risk-label').style.color = getRiskColor(combinedScore);
    document.getElementById('risk-detail').textContent =
      `${trackerCount} tracker${trackerCount !== 1 ? 's' : ''} detected · ${urlResult.label} URL`;

    updateRing(combinedScore);

    // Phishing model card
    const phishBadge = document.getElementById('phishing-badge');
    phishBadge.textContent = urlResult.label;
    phishBadge.className = `model-badge ${getBadgeClass(urlResult.label)}`;
    document.getElementById('phishing-bar').style.width = `${urlResult.confidence}%`;
    document.getElementById('phishing-bar').style.background = getRiskColor(urlResult.risk_score);
    document.getElementById('phishing-meta').textContent =
      `Confidence ${urlResult.confidence}% · DistilBERT`;

    // Policy card placeholder
    const policyBadge = document.getElementById('policy-badge');
    policyBadge.textContent = trackerCount > 3 ? 'RISKY' : 'SAFE';
    policyBadge.className = `model-badge ${trackerCount > 3 ? 'badge-risky' : 'badge-safe'}`;
    document.getElementById('policy-bar').style.width = `${trackerCount > 3 ? 65 : 20}%`;
    document.getElementById('policy-meta').textContent = 'DistilBERT · Based on tracker analysis';

    // Tracker tags
    const tagsContainer = document.getElementById('tracker-tags');
    tagsContainer.innerHTML = '';
    const tagClasses = { ads: 'tag-ads', analytics: 'tag-analytics', fingerprint: 'tag-finger', social: 'tag-social' };
    let anyTracker = false;
    for (const [cat, count] of Object.entries(trackers)) {
      if (count > 0) {
        anyTracker = true;
        const tag = document.createElement('span');
        tag.className = `tag ${tagClasses[cat]}`;
        tag.textContent = `${cat.charAt(0).toUpperCase() + cat.slice(1)} ×${count}`;
        tagsContainer.appendChild(tag);
      }
    }
    if (!anyTracker) {
      tagsContainer.innerHTML = '<span class="tag tag-social">No trackers found ✓</span>';
    }

    // Explanation
    let explanation = '';
    if (urlResult.is_phishing) {
      explanation = `This URL shows phishing patterns (${urlResult.confidence}% confidence). Do not enter personal information on this site.`;
    } else if (combinedScore > 50) {
      explanation = `This site has ${trackerCount} active trackers collecting your data. ${trackerCount > 3 ? 'Heavy advertising and analytics presence detected.' : 'Moderate data collection in progress.'}`;
    } else {
      explanation = `This site appears relatively safe. ${trackerCount > 0 ? 'Minor tracker activity detected.' : 'No significant privacy concerns found.'}`;
    }
    document.getElementById('explanation-text').textContent = explanation;

    // Timestamp
    document.getElementById('scan-time').textContent =
      `Scanned at ${new Date().toLocaleTimeString()}`;

    // Save to history
    saveToHistory({ url, score: combinedScore, label: urlResult.label, time: new Date().toISOString() });

  } catch (err) {
    document.getElementById('loading-state').style.display = 'none';
    document.getElementById('error-state').style.display = 'block';
    document.getElementById('error-state').textContent = `Error: ${err.message}. Is the API running?`;
    console.error('PrivAware error:', err);
  }
}

// Init
document.addEventListener('DOMContentLoaded', () => {
  chrome.tabs.query({ active: true, currentWindow: true }, (tabs) => {
    const url = tabs[0]?.url || '';
    if (url.startsWith('chrome://') || url.startsWith('chrome-extension://')) {
      document.getElementById('loading-state').style.display = 'none';
      document.getElementById('error-state').style.display = 'block';
      document.getElementById('error-state').textContent = 'Cannot scan browser internal pages.';
      return;
    }
    scanPage(url);
  });

  document.getElementById('rescan-btn').addEventListener('click', () => {
    chrome.tabs.query({ active: true, currentWindow: true }, (tabs) => {
      scanPage(tabs[0]?.url || '');
    });
  });
});
'''

with open('/content/privaware-extension/popup.js', 'w') as f:
    f.write(popup_js)
print('popup.js written!')

## Cell 5 — Write background.js and content.js

In [ ]:
background_js = '''chrome.runtime.onInstalled.addListener(() => {
  console.log('PrivAware v2 installed');
  chrome.storage.local.set({ scanHistory: [] });
});
'''

content_js = '''// Content script - runs on every page
// Currently just signals the page has loaded
console.log('PrivAware content script loaded on:', window.location.href);
'''

with open('/content/privaware-extension/background.js', 'w') as f:
    f.write(background_js)

with open('/content/privaware-extension/content.js', 'w') as f:
    f.write(content_js)

print('background.js written!')
print('content.js written!')

## Cell 6 — Generate extension icons programmatically

In [ ]:
from PIL import Image, ImageDraw
import os

def create_icon(size):
    img = Image.new('RGBA', (size, size), (0, 0, 0, 0))
    draw = ImageDraw.Draw(img)
    # Purple rounded background
    draw.rounded_rectangle([0, 0, size-1, size-1], radius=size//5, fill=(108, 99, 255, 255))
    # Shield shape
    cx, cy = size // 2, size // 2
    s = size // 3
    draw.polygon([
        (cx, cy - s),
        (cx + s, cy - s//2),
        (cx + s, cy + s//3),
        (cx, cy + s),
        (cx - s, cy + s//3),
        (cx - s, cy - s//2)
    ], fill=(255, 255, 255, 230))
    return img

for size in [16, 48, 128]:
    icon = create_icon(size)
    icon.save(f'/content/privaware-extension/icons/icon{size}.png')
    print(f'icon{size}.png created')

print('All icons created!')

## Cell 7 — Zip and download the extension

In [ ]:
import shutil
from google.colab import files

# Zip the extension folder
shutil.make_archive('/content/privaware-extension', 'zip', '/content/privaware-extension')

# Also save to Drive
from google.colab import drive
drive.mount('/content/drive')
shutil.copy('/content/privaware-extension.zip', '/content/drive/MyDrive/privaware-v2/privaware-extension.zip')

print('Extension zipped and saved to Drive!')
print('\nDownloading to your Mac now...')
files.download('/content/privaware-extension.zip')
print('\nDownload started!')

## Cell 8 — How to load the extension in Chrome

Run this cell to print the exact steps.

In [ ]:
print('='*55)
print('HOW TO LOAD THE EXTENSION IN CHROME')
print('='*55)
print()
print('1. Unzip privaware-extension.zip on your Mac')
print('   Double click the zip file -> a folder appears')
print()
print('2. Open Chrome and go to:')
print('   chrome://extensions')
print()
print('3. Top right corner -> toggle ON Developer mode')
print()
print('4. Click Load unpacked')
print('   Select the privaware-extension folder (not the zip)')
print()
print('5. PrivAware should appear in your extensions list')
print('   Pin it to toolbar: puzzle icon -> pin PrivAware')
print()
print('6. Visit any website -> click the PrivAware icon')
print('   You should see the risk score popup!')
print()
print('7. Test these sites:')
print('   forbes.com       -> should be HIGH risk')
print('   github.com       -> should be LOW risk')
print('   nytimes.com      -> should be MODERATE risk')
print()
print('If the popup shows an error, make sure:')
print('  -> HuggingFace Space is Running (not Building)')
print('  -> Space URL: https://V3d4nt7-privaware-api.hf.space')
print()
print('='*55)
print('DAY 5 COMPLETE! Message me for Day 6 (landing page).')
print('='*55)